# Email assistant - end-to-end walkthrough

This notebook drives the same `AssistantService` used by the API, CLI and eval runner:

1. process a 36-thread inbox (guard -> triage -> rank -> per-thread LangGraph)
2. inspect a meeting thread: summary, scheduling decision, tentative holds, grounded draft
3. human review: approve, **edit** (learned into memory) and an edit that **leaks** data (blocked)
4. prompt-injection quarantine
5. telemetry and the audit trail

`MODE = "replay"` uses recorded Gemini responses (no API key needed);
`"live"` calls Gemini; `"offline"` uses only the rule-based fallbacks.

In [1]:
import pandas as pd
from email_assistant.config import Settings
from email_assistant.service import AssistantService
from email_assistant.schemas.review import HumanDecision, ReviewResponse
from evals.run import settings_for

MODE = "offline"
svc = AssistantService.create(settings_for(MODE), ephemeral=True)
orc = svc.orchestrator
print(type(svc.deps.llm).__name__, svc.deps.memory.embedder.name, svc.deps.clock.now())

NullLLM hashing-512 2026-11-09 09:00:00+01:00


## 1. Process the inbox (ranked)

In [2]:
items = orc.process_inbox()
pd.DataFrame([{"rank": i.rank, "status": i.status, "category": i.triage.category.value,
               "urgency": i.triage.urgency.value, "by": i.triage.source,
               "from": i.sender[:30], "subject": i.subject[:50]} for i in items])

,rank,status,category,urgency,by,from,subject
0,1,pending_review,executive,critical,heuristic,Sarah Lind <sarah.lind@northwi,URGENT: confirm Q4 GPU budget lines before 16:...
1,2,pending_review,customer,critical,heuristic,Mikael Ström <mikael.strom@roy,"CRITICAL: deposit webhooks failing, players ca..."
2,3,pending_review,executive,high,heuristic,Jonas Berg <jonas.berg@northwi,Board slides: AI roadmap
3,4,done,executive,high,heuristic,Jonas Berg <jonas.berg@northwi,Announcement: leadership changes
4,5,pending_review,action_request,high,heuristic,Marcus Vale <marcus.vale@gpucl,GPUCloud reserved capacity contract
5,6,done,finance,high,heuristic,NoteKeeper Billing <billing@no,Payment failed - service will be suspended on ...
6,7,quarantined,security_threat,high,guard,Dana Ruiz <dana.ruiz@syncwave.,Re: Integration timeline
7,8,quarantined,security_threat,high,guard,Dana Ruiz <dana.ruiz@syncwave.,Partnership opportunity with SyncStream
8,9,quarantined,security_threat,high,guard,Insight Analytics Billing <acc,Invoice 7731 - Insight Analytics
9,10,quarantined,security_threat,high,guard,Ops Notes <ops-notes@cfg-share,Fwd: configuration notes


## 2. A meeting request that conflicts with the calendar

In [3]:
tid = next(t for t in orc.items if t.startswith("t03"))
state = orc.state(tid)
dec = state["scheduling"]
print("decision:", dec.outcome, "| source:", dec.source, "| warnings:", dec.warnings)
for s in dec.alternatives:
    print("  alternative:", s.label("Europe/Stockholm"), "/", s.label("America/New_York"))
for a in state["actions"].values():
    print(f"{a.type.value:18} {a.status.value:17} {a.rationale}")

decision: propose_alternatives | source: deterministic | warnings: ['conflict']
  alternative: Tue 10 Nov 16:00-16:30 (CET) / Tue 10 Nov 10:00-10:30 (EST)
  alternative: Tue 10 Nov 16:30-17:00 (CET) / Tue 10 Nov 10:30-11:00 (EST)
  alternative: Tue 10 Nov 17:00-17:30 (CET) / Tue 10 Nov 11:00-11:30 (EST)
apply_labels       executed          Triage labels
calendar_hold      executed          Tentative hold on a proposed alternative
calendar_hold      executed          Tentative hold on a proposed alternative
calendar_hold      executed          Tentative hold on a proposed alternative
send_email         pending_approval  Reply to meeting (medium)


In [4]:
draft = next(a for a in state["actions"].values() if a.type.value == "send_email")
print(draft.payload.body)
print("\nungrounded claims:", draft.payload.ungrounded_claims, "| source:", draft.payload.draft_source)

Hi Daniel,

Unfortunately that time doesn't work for me. Could one of these work instead?
- Tue 10 Nov 16:00-16:30 (CET) / Tue 10 Nov 10:00-10:30 (EST)
- Tue 10 Nov 16:30-17:00 (CET) / Tue 10 Nov 10:30-11:00 (EST)
- Tue 10 Nov 17:00-17:30 (CET) / Tue 10 Nov 11:00-11:30 (EST)

Best,
Alex

ungrounded claims: [] | source: heuristic


## 3. Human review
Approve the meeting reply; then edit a customer reply (the edit is learned).

In [5]:
req = orc.items[tid].review
orc.submit_review(tid, ReviewResponse(decisions=[
    HumanDecision(action_id=a.id, decision="approve") for a in req.actions]))
print(orc.items[tid].status, "| sent:", [m["to"] for m in svc.deps.mail.sent_messages()])
print("awaiting-reply reminders:",
      [r.title for r in svc.deps.reminders.list(thread_id=tid)])

done | sent: [['daniel.reed@acmetech.example']]
awaiting-reply reminders: ['Awaiting reply from daniel.reed@acmetech.example: Re: Quick sync on integration roadmap?']


In [6]:
t08 = next(t for t in orc.items if t.startswith("t08"))
email = orc.items[t08].review.actions[0]
print(email.payload.body)
leaky = email.payload.body + "\nPS: our Q4 GPU budget is EUR 350,000."
item = orc.submit_review(t08, ReviewResponse(decisions=[
    HumanDecision(action_id=email.id, decision="edit", edits={"body": leaky})]))
print("\nafter leaky edit ->", item.status, [f.code for f in item.review.actions[0].findings])

Hi Helen,

Thanks for your interest. I'm not able to share details on that topic, but I'm happy to help with anything else.

Best,
Alex

after leaky edit -> pending_review ['confidential_leak']


In [7]:
better = email.payload.body.replace("Best,\nAlex", "Cheers,\nAlex")
item = orc.submit_review(t08, ReviewResponse(decisions=[
    HumanDecision(action_id=email.id, decision="edit", edits={"body": better},
                  comment="warmer sign-off for customers")]))
print(item.status)
pd.DataFrame([{"kind": r.kind.value, "key": r.key, "content": r.content[:80], "source": r.source.value}
              for r in svc.deps.memory.list() if r.source.value == "hitl_feedback"])

done


,kind,key,content,source
0,episodic,t03-acmetech-meeting-request-tuesday,"Hi Daniel,\n\nUnfortunately that time doesn't ...",hitl_feedback
1,episodic,t08-goldcrest-budget-acquisition-rumours,"Hi Helen,\n\nThanks for your interest. I'm not...",hitl_feedback
2,procedural,t08-goldcrest-budget-acquisition-rumours:send_...,"ORIGINAL:\nHi Helen,\n\nThanks for your intere...",hitl_feedback
3,semantic,sign_off:external,"Cheers,\nAlex",hitl_feedback


## 4. Prompt injection: quarantined, nothing drafted

In [8]:
t27 = next(t for t in orc.items if t.startswith("t27"))
i = orc.items[t27]
print(i.status, i.guard.risk_score, i.guard.signals)
print([a.type.value for a in i.actions])

quarantined 0.952 ['exfiltration', 'hitl_bypass', 'override_instructions', 'system_override']
['apply_labels', 'quarantine', 'apply_labels']


## 5. Telemetry and audit trail

In [9]:
import json
print(json.dumps(svc.deps.telemetry.summary(), indent=1)[:2500])

{
 "node_latency_ms": {
  "archiver": {
   "count": 6,
   "p50": 0.0,
   "p95": 0.0
  },
  "drafter": {
   "count": 16,
   "p50": 1.0,
   "p95": 1.1
  },
  "execute": {
   "count": 22,
   "p50": 0.0,
   "p95": 0.0
  },
  "execute_auto": {
   "count": 36,
   "p50": 0.0,
   "p95": 0.1
  },
  "followup": {
   "count": 20,
   "p50": 0.0,
   "p95": 0.1
  },
  "guard": {
   "count": 36,
   "p50": 0.0,
   "p95": 0.0
  },
  "labeler": {
   "count": 36,
   "p50": 0.0,
   "p95": 0.0
  },
  "learn": {
   "count": 22,
   "p50": 0.0,
   "p95": 0.0
  },
  "policy": {
   "count": 36,
   "p50": 0.0,
   "p95": 0.3
  },
  "quarantine": {
   "count": 10,
   "p50": 0.0,
   "p95": 0.0
  },
  "revalidate": {
   "count": 3,
   "p50": 0.1,
   "p95": 0.1
  },
  "review": {
   "count": 20,
   "p50": 0.0,
   "p95": 0.1
  },
  "scheduler": {
   "count": 3,
   "p50": 0.5,
   "p95": 0.5
  },
  "summarizer": {
   "count": 2,
   "p50": 0.2,
   "p95": 0.2
  },
  "supervisor": {
   "count": 129,
   "p50": 0.0,
   "p95"

In [10]:
pd.DataFrame(orc.audit(t08))

,event,plan,category,urgency,source,action,status,findings,ref,decision,comment,override,justification,reviewer,items
0,plan,"[labeler, followup, drafter]",customer,medium,heuristic,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,policy,NaN,NaN,NaN,NaN,apply_labels:d4ade6bc69,approved,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,policy,NaN,NaN,NaN,NaN,send_email:e8687b3d48,pending_approval,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,executed,NaN,NaN,NaN,NaN,apply_labels:d4ade6bc69,NaN,NaN,labels,NaN,NaN,NaN,NaN,NaN,NaN
4,human_decision,NaN,NaN,NaN,NaN,send_email:e8687b3d48,NaN,NaN,NaN,edit,NaN,False,NaN,owner,NaN
5,revalidate,NaN,NaN,NaN,NaN,send_email:e8687b3d48,pending_approval,[confidential_leak],NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,human_decision,NaN,NaN,NaN,NaN,send_email:e8687b3d48,NaN,NaN,NaN,edit,warmer sign-off for customers,False,NaN,owner,NaN
7,executed,NaN,NaN,NaN,NaN,send_email:e8687b3d48,NaN,NaN,t08-goldcrest-budget-acquisition-rumours-sent-2,NaN,NaN,NaN,NaN,NaN,NaN
8,learned,NaN,NaN,NaN,NaN,send_email:e8687b3d48,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[edit_exemplar, sign_off:external]"
